# Choosing metrics based on business cost

_Doing ML for Real — the skills that matter_

**Accuracy doesn't pay the bills. Pick the threshold that minimizes real dollars.**

A classifier outputs a score; a decision is what you do with it. The bridge between them is a threshold $t$: act if score $\ge t$, otherwise don't.

       Different thresholds trade one kind of mistake for the other. Lower $t$ ⇒ more alerts, fewer misses (fewer FN, more FP). Higher $t$ ⇒ the reverse.

---

This notebook is a practice scaffold for the **Choosing metrics based on business cost** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, precision_recall_curve, brier_score_loss
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

# positive class = malignant tumor (the costly miss). sklearn target: 0=malignant, 1=benign.
data = load_breast_cancer()
X = data.data
y = (data.target == 0).astype(int)          # 1 = malignant (positive)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.4, random_state=0, stratify=y)

clf = make_pipeline(StandardScaler(),
                    LogisticRegression(max_iter=5000)).fit(X_tr, y_tr)
p = clf.predict_proba(X_te)[:, 1]           # P(malignant)

# --- 1) the cost / utility matrix: a missed malignant tumor is far worse ---
C_FP = 1.0     # false alarm: benign flagged malignant (extra biopsy)
C_FN = 20.0    # MISS: malignant called benign (could be fatal)

# --- 2) expected cost at each threshold, straight from the confusion matrix ---
def expected_cost(y_true, scores, t, c_fp, c_fn):
    y_hat = (scores >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0, 1]).ravel()
    return c_fp * fp + c_fn * fn

# --- 3) pick the threshold that MINIMIZES expected cost (not the default 0.5) ---
# use the candidate thresholds the PR curve already evaluated
_, _, thresholds = precision_recall_curve(y_te, p)
costs = [expected_cost(y_te, p, t, C_FP, C_FN) for t in thresholds]
t_best = thresholds[int(np.argmin(costs))]

t_star = C_FP / (C_FP + C_FN)               # Bayes-optimal threshold (if calibrated)
print(f"cost-minimizing threshold: {t_best:.3f}")
print(f"Bayes-optimal t*:          {t_star:.3f}")
print(f"cost at t_best: {min(costs):.0f}   "
      f"cost at 0.5: {expected_cost(y_te, p, 0.5, C_FP, C_FN):.0f}")

# --- 4) precision@k for a capacity-limited review team (top-k highest scores) ---
k = 20
top_k = np.argsort(p)[::-1][:k]
print(f"precision@{k}: {y_te[top_k].mean():.3f}")

# --- 5) calibrate the probabilities so the threshold means what it says ---
cal = CalibratedClassifierCV(clf, method="isotonic", cv=5).fit(X_tr, y_tr)
p_cal = cal.predict_proba(X_te)[:, 1]
frac_pos, mean_pred = calibration_curve(y_te, p_cal, n_bins=10)

# --- 6) report the metrics tied to the decision, not just accuracy/AUC ---
print(f"Brier (raw):        {brier_score_loss(y_te, p):.4f}")
print(f"Brier (calibrated): {brier_score_loss(y_te, p_cal):.4f}")
print("reliability (pred vs actual):",
      list(zip(np.round(mean_pred, 2), np.round(frac_pos, 2))))

## Visualize the data & results

_On the breast-cancer data, where does expected cost bottom out, and how far is that from the naive 0.5 threshold?_

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# positive class = malignant (sklearn target: 0=malignant, 1=benign)
data = load_breast_cancer()
X = data.data
y = (data.target == 0).astype(int)          # 1 = malignant (the costly miss)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.4, random_state=0, stratify=y)

# a deliberately mild model (C=0.05) so the cost curve has a clear interior minimum
clf = make_pipeline(StandardScaler(),
                    LogisticRegression(C=0.05, max_iter=5000)).fit(X_tr, y_tr)
p = clf.predict_proba(X_te)[:, 1]

C_FP, C_FN = 1.0, 10.0                        # missing a malignancy is 10x a false alarm
thresholds = [0.05, 0.10, 0.13, 0.18, 0.24, 0.30, 0.36,
              0.42, 0.50, 0.58, 0.66, 0.74, 0.82, 0.90]
costs = []
for t in thresholds:
    y_hat = (p >= t).astype(int)
    fp = int(((y_hat == 1) & (y_te == 0)).sum())
    fn = int(((y_hat == 0) & (y_te == 1)).sum())
    costs.append(C_FP * fp + C_FN * fn)

best = thresholds[int(np.argmin(costs))]
print("threshold -> cost:", list(zip(thresholds, costs)))
print("cost-minimizing threshold:", best, " cost:", min(costs))
print("naive 0.5 cost:", costs[thresholds.index(0.50)])
print("Bayes-optimal t*:", round(C_FP / (C_FP + C_FN), 3))

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A spam filter is graded at 99.1% accuracy and ships with the default 0.5 threshold. A product manager says "great, ship it." The cost of a missed spam (FN) is one annoying email; the cost of a real message sent to spam (FP) is a lost customer reply. Is accuracy the right call, and is 0.5 the right threshold?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify the two error costs. — _Here a False Positive (good mail marked spam) is far more expensive than a False Negative (spam in the inbox) — the costs are very asymmetric, the opposite of the fraud case._
- Note the base rate. — _Spam is common, so 99.1% accuracy can be reached by a model that's reckless about FPs. Accuracy hides the error that actually hurts._
- Set the threshold from the cost ratio. — _With FP much costlier than FN, $t^\*=\frac{C_{\text{FP}}}{C_{\text{FP}}+C_{\text{FN}}}$ is well above 0.5 — be conservative about flagging spam._

**Answer:** No on both counts. Accuracy is the wrong metric (FP and FN costs differ wildly), and 0.5 is the wrong threshold. Because a False Positive costs far more than a False Negative, $t^\*$ should be pushed above 0.5 so the filter only flags mail it's very sure about. Report expected cost (or precision on the "spam" decision), not accuracy.

</details>

**Problem 2.** You have a fitted gradient-boosted model. The cost matrix says a miss is 8× a false alarm. You compute $t^\*=1/(1+8)=0.111$ and ship that cutoff. The realized cost in production is much worse than predicted. What's the most likely cause, and how do you check it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what $t^\*$ assumes. — _The $t^\*$ formula is only valid if the model's scores are true probabilities — a 0.111 score must mean an 11.1% chance._
- Suspect calibration. — _Gradient-boosted trees output mis-scaled scores; "0.111" may correspond to a very different real probability, so the cutoff lands in the wrong place._
- Calibrate and re-derive. — _Wrap the model in CalibratedClassifierCV (isotonic or sigmoid), check calibration_curve / Brier / ECE, then re-pick the threshold — or skip $t^\*$ and sweep the empirical cost curve directly._

**Answer:** The scores are uncalibrated, so $t^\*=0.111$ doesn't correspond to the intended 8:1 trade-off. Fix it with CalibratedClassifierCV and verify with a reliability curve (Brier score / ECE). Alternatively, bypass the formula and pick the threshold that minimizes the empirical expected-cost curve on a validation split — that works even on raw scores, though calibration is still wise for interpretability.

</details>

**Problem 3.** A fraud team can manually review only 50 cases per day, but the model scores 40,000 transactions daily. The DS lead wants to "optimize the threshold for expected cost." What metric actually governs this setup, and how do you compute it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Spot the capacity constraint. — _The action (manual review) is capped at $k=50$. Thresholds that surface 4,000 alerts are irrelevant — only the top 50 get acted on._
- Switch to precision@k. — _Rank all 40,000 by score, take the top $k=50$, and measure precision there: what fraction of those 50 are true fraud. That is the metric the team feels._
- Tie it to value. — _Multiply precision@50 by 50 to get expected true frauds caught per day, times the dollars saved per catch — the business headline._

**Answer:** This is a capacity-limited action, so the governing metric is precision@k with $k=50$ — not a global cost-minimizing threshold. Rank transactions by score, take the top 50, and report the fraction that are actually fraud (and the resulting dollars saved). A threshold sweep is the wrong frame here because the team's budget, not a cutoff, decides how many cases get acted on.

</details>